In [1]:
import ee
import geemap

ee.Initialize(project="oc-flux")

Map = geemap.Map()

In [ ]:
Load HydroRIVERS

In [2]:
rivers = ee.FeatureCollection(
    "projects/oc-flux/assets/HydroRIVERS_v10_asi_shp"
)

In [ ]:
Define Study Area

Use the Cauvery point we have been using.

In [3]:
roi = ee.Geometry.Point(
    [79.8033,11.1313]
).buffer(10000)

In [ ]:
Filter Rivers

In [4]:
cauvery = rivers.filterBounds(roi)

In [ ]:
Display

In [5]:
Map.centerObject(roi,11)

Map.addLayer(
    roi,
    {"color":"red"},
    "ROI"
)

Map.addLayer(
    cauvery,
    {"color":"blue"},
    "River"
)

Map

Map(center=[11.131310258873006, 79.80330028994707], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
Count River Segments

In [6]:
print(
    "River Segments:",
    cauvery.size().getInfo()
)

River Segments: 20


In [ ]:
Inspect Attributes

In [7]:
print(
    cauvery.first().propertyNames().getInfo()
)

['MAIN_RIV', 'UPLAND_SKM', 'ORD_CLAS', 'LENGTH_KM', 'CATCH_SKM', 'ORD_STRA', 'DIST_DN_KM', 'ORD_FLOW', 'NEXT_DOWN', 'DIST_UP_KM', 'HYRIV_ID', 'DIS_AV_CMS', 'ENDORHEIC', 'HYBAS_L12', 'system:index']


In [ ]:
Find the Largest River Segment

Let's sort by average discharge.

In [8]:
largest = cauvery.sort(
    "DIS_AV_CMS",
    False
)

mainRiver = ee.Feature(
    largest.first()
)

In [ ]:
Display Main Channel
You should now see

Blue = all river segments
Yellow = largest/main river segment near your ROI

In [10]:
Map.addLayer(
    mainRiver,
    {"color":"yellow"},
    "Main River"
)

Map

Map(bottom=246130.0, center=[11.131310258873006, 79.80330028994707], controls=(WidgetControl(options=['positio…

In [ ]:
Check the Discharge
Look for

DIS_AV_CMS

LENGTH_KM

ORD_FLOW

In [11]:
print(
    mainRiver.getInfo()["properties"]
)

{'CATCH_SKM': 5.9, 'DIST_DN_KM': 0, 'DIST_UP_KM': 71.8, 'DIS_AV_CMS': 8.754, 'ENDORHEIC': 0, 'HYBAS_L12': 4120028820, 'HYRIV_ID': 41378923, 'LENGTH_KM': 0.91, 'MAIN_RIV': 41378923, 'NEXT_DOWN': 0, 'ORD_CLAS': 1, 'ORD_FLOW': 6, 'ORD_STRA': 3, 'UPLAND_SKM': 649.7}


In [ ]:
Get the Geometry
This line geometry is what we'll use to generate sampling points.

In [12]:
geometry = mainRiver.geometry()